# Task 12: GHZ state prep via cluster fusion

Demonstrates the full DQC loop for `ghz_via_fusion`:

1. `bell_basis_circuit` — Bell measurement pre-rotation
2. `ghz_cluster_prep_circuit` — GHZ state on offset/independent sites
3. **DQC loop step-by-step**: measure → `decoder(outcome)` → correction circuit → pure output state
4. Statistical verification: fidelity = 1.0 for all outcomes, all 4 outcomes covered
5. Multi-boundary case (n=6, C=4 clusters, 3 fusions)

In [ ]:
import numpy as np
from qaravan.applications import (
    bell_basis_circuit,
    ghz_cluster_prep_circuit,
    ghz_fusion_decoder,
    ghz_via_fusion,
)
from qaravan.backends.statevector import Statevector

INV_SQRT2 = 1.0 / np.sqrt(2)

## 1. Bell measurement pre-rotation

`bell_basis_circuit(a, b, n)` = CNOT(a→b), H(a).  
Applied to the 4 Bell states it returns the 4 computational basis states.

In [ ]:
bell_states = {
    "|Φ+⟩": np.array([INV_SQRT2, 0, 0, INV_SQRT2]),
    "|Φ-⟩": np.array([INV_SQRT2, 0, 0, -INV_SQRT2]),
    "|Ψ+⟩": np.array([0, INV_SQRT2, INV_SQRT2, 0]),
    "|Ψ-⟩": np.array([0, INV_SQRT2, -INV_SQRT2, 0]),
}

circ = bell_basis_circuit(0, 1, 2)
print("bell_basis_circuit(0,1,2):", circ.gates)
print()
for name, arr in bell_states.items():
    result = Statevector(array=arr).apply(circ)
    print(f"{name} → {result}")

## 2. Two independent GHZ clusters (pre-fusion)

Star-topology GHZ on offset sites. Purity of each cluster = 1 confirms they are
unentangled with each other before any fusion.

In [ ]:
init6 = Statevector(bitstring="000000")
sv_clusters = init6.apply(
    ghz_cluster_prep_circuit([0, 1, 2], 6) + ghz_cluster_prep_circuit([3, 4, 5], 6)
)
print("State before fusion:")
print(sv_clusters)
print()
for sites in [[0, 1, 2], [3, 4, 5]]:
    rho = sv_clusters.rdm(sites)
    purity = float(np.real(np.trace(rho @ rho)))
    print(f"Purity of cluster on {sites}: {purity:.6f}  (1.0 → pure, independent)")

## 3. DQC loop step-by-step — 4-qubit GHZ (n=4, k=3)

Protocol: 2 clusters × 3 qubits = 6 physical qubits. One fusion at boundary (q_L=2, q_R=3).
Kept sites: {0,1} from cluster 0, {4,5} from cluster 1.

**Re-run this cell several times** — each run draws a different outcome and applies the matching correction.

In [ ]:
n, k = 4, 3
C = (n - 2) // (k - 2)   # = 2
total_qubits = C * k      # = 6
q_L, q_R = 2, 3           # fusion sites at this boundary
kept = [0, 1, 4, 5]

# ── Step 1: prepare clusters ──────────────────────────────────────────────
sv = Statevector(bitstring="000000").apply(
    ghz_cluster_prep_circuit([0, 1, 2], 6) + ghz_cluster_prep_circuit([3, 4, 5], 6)
)
print("After cluster prep:")
print(sv)

# ── Step 2: Bell pre-rotation + measurement ───────────────────────────────
sv = sv.apply(bell_basis_circuit(q_L, q_R, total_qubits))
sv_post_meas, outcome = sv.measure_and_collapse([q_L, q_R])
print(f"\nMeasurement outcome on qubits {{{q_L},{q_R}}}: '{outcome}'")
print("Post-measurement state on active qubits:")
print(sv_post_meas.drop_sites([q_L, q_R], outcome))

# ── Step 3: decoder → correction circuit ─────────────────────────────────
decoder = ghz_fusion_decoder(j=0, k=k, C=C, total_qubits=total_qubits)
correction = decoder(outcome)
gate_str = str(correction.gates) if correction.gates else "[none needed]"
print(f"\ndecoder('{outcome}') → correction: {gate_str}")

# ── Step 4: apply correction → final GHZ state ───────────────────────────
sv_final = sv_post_meas.apply(correction) if correction.gates else sv_post_meas
print("\nFinal state on kept qubits {0,1,4,5}:")
print(sv_final.drop_sites([q_L, q_R], outcome))

# ── Verify ────────────────────────────────────────────────────────────────
ghz4 = np.zeros(16); ghz4[0] = ghz4[15] = INV_SQRT2
rdm = sv_final.rdm(kept)
fidelity = float(np.real(ghz4 @ rdm @ ghz4))
print(f"\nFidelity with |GHZ_4⟩: {fidelity:.12f}")

## 4. Statistical verification: all outcomes, always fidelity = 1.0

In [ ]:
from collections import Counter

outcome_counts = Counter()
fidelities = []
ghz4 = np.zeros(16); ghz4[0] = ghz4[15] = INV_SQRT2

for _ in range(200):
    sv_f, outcomes = ghz_via_fusion(4, k=3)
    rdm = sv_f.rdm([0, 1, 4, 5])
    fidelities.append(float(np.real(ghz4 @ rdm @ ghz4)))
    outcome_counts[outcomes[0]] += 1

print("All fidelities = 1.0:", all(abs(f - 1.0) < 1e-9 for f in fidelities))
print("Min fidelity:          ", min(fidelities))
print()
print("Outcome distribution (200 runs):")
for o in sorted(outcome_counts):
    print(f"  {o}: {outcome_counts[o]}")

## 5. Multi-boundary: 6-qubit GHZ (n=6, k=3)

C=4 clusters, 3 sequential fusions, 12 physical qubits.  
Kept sites: {0,1} | {4} | {7} | {10,11}

In [ ]:
kept6 = [0, 1, 4, 7, 10, 11]
ghz6 = np.zeros(64); ghz6[0] = ghz6[63] = INV_SQRT2

fidelities6 = []
for _ in range(20):
    sv_f, outcomes = ghz_via_fusion(6, k=3)
    assert len(outcomes) == 3
    rdm = sv_f.rdm(kept6)
    fidelities6.append(float(np.real(ghz6 @ rdm @ ghz6)))

print("All fidelities = 1.0 (n=6, 3 fusions):", all(abs(f - 1.0) < 1e-9 for f in fidelities6))
print("Example 3-outcome sequence:", outcomes)